# Buổi 2 — Khoảng tin cậy & t-test với dữ liệu trường học PISA VN

In [1]:
# Chạy ô này đầu tiên. Dữ liệu (PISA 2025, Việt Nam) được tải trực tiếp từ GitHub ở ô kế tiếp — không cần tải/upload tay.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf, statsmodels.api as sm
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"]=(7,4); pd.set_option("display.precision",3)

#### 📥 Đầu vào

nạp thư viện (`numpy, pandas, scipy.stats, statsmodels`).

In [2]:
# ---- TẢI DỮ LIỆU (2 bảng: cấp trường 195 dòng, cấp học sinh 7.368 dòng) ----
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
hs = pd.read_csv(RAW + "vnm_hocsinh_7368.csv")
VUNG={1:"ĐB sông Cửu Long",2:"Bắc TB, DH TB & Tây Nguyên",3:"Trung du & MN phía Bắc",4:"ĐB sông Hồng",5:"Đông Nam Bộ"}
truong["vung_ten"]=truong.vung.map(VUNG); truong["loai"]=truong.PRIVATESCH.map({1:"Công",2:"Tư"})
W=["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP","AVLRSOFT","ENCOURPG"]
print("Bảng trường:",truong.shape,"| Bảng học sinh:",hs.shape)

Bảng trường: (195, 36) | Bảng học sinh: (7368, 54)


#### 📥 Đầu vào

tải lại 2 bảng PISA VN từ GitHub (giống notebook buổi 1 — mỗi notebook độc lập, tự chạy được).

#### 📤 Đầu ra thật

`(195, 36)` và `(7368, 54)` — khớp buổi 1. ✅

## 1. Khoảng tin cậy 95%

In [3]:
def ci(x):
    x=x.dropna(); se=stats.sem(x); l,u=stats.t.interval(.95,len(x)-1,loc=x.mean(),scale=se); return dict(n=len(x),mean=x.mean(),sd=x.std(),se=se,lo=l,hi=u)
pd.DataFrame({"EDULEAD":ci(truong.EDULEAD),"sci_mean":ci(truong.sci_mean)}).T

,n,mean,sd,se,lo,hi
EDULEAD,195.0,0.765,0.899,0.064,0.638,0.892
sci_mean,195.0,0.446,0.090,0.006,0.433,0.459


#### 📥 Đầu vào

2 biến cấp trường `EDULEAD` (lãnh đạo giáo dục, WLE) và `sci_mean` (điểm Khoa học TB trường) — tính khoảng tin cậy 95% cho trung bình mỗi biến.

#### 📤 Đầu ra thật

`EDULEAD`: mean=0,765, CI95%=[0,638; 0,892] (khoảng khá rộng, ±0,13, vì SD=0,899 lớn so với n=195); `sci_mean`: mean=0,446, CI95%=[0,433; 0,459] (khoảng RẤT HẸP, ±0,013) — vì `sci_mean` có SD chỉ 0,09 (đã "làm mượt" do là trung bình cấp trường của nhiều học sinh, xem giải thích ở notebook buổi 1). ✅ Hợp lý: biến càng ít biến thiên (SD nhỏ) thì CI càng hẹp/chính xác hơn ở cùng cỡ mẫu.

## 2. t độc lập: trường công (161) vs tư (34) — *đọc cả hai dòng: phương sai bằng nhau và Welch*

In [4]:
def tt(v):
    a=truong[truong.PRIVATESCH==1][v]; b=truong[truong.PRIVATESCH==2][v]
    lev=stats.levene(a,b,center="mean"); pooled=stats.ttest_ind(a,b); welch=stats.ttest_ind(a,b,equal_var=False)
    sp=np.sqrt(((len(a)-1)*a.var()+(len(b)-1)*b.var())/(len(a)+len(b)-2))
    return dict(TB_cong=a.mean(),TB_tu=b.mean(),chenh=a.mean()-b.mean(),Levene_p=lev.pvalue,t_pooled=pooled.statistic,p_pooled=pooled.pvalue,t_Welch=welch.statistic,p_Welch=welch.pvalue,d=(a.mean()-b.mean())/sp)
pd.DataFrame({v:tt(v) for v in ["EDULEAD","NEGSCLIM","STAFFSHORT","sci_mean"]}).T

,TB_cong,TB_tu,chenh,Levene_p,t_pooled,p_pooled,t_Welch,p_Welch,d
EDULEAD,0.696,1.091,-0.395,0.148,-2.356,0.019,-1.874,0.068,-0.445
NEGSCLIM,-0.298,0.194,-0.492,0.530,-1.965,0.051,-1.802,0.078,-0.371
STAFFSHORT,0.142,-0.147,0.289,0.187,1.469,0.143,1.329,0.191,0.277
sci_mean,0.449,0.432,0.017,0.685,0.986,0.325,1.032,0.307,0.186


**❓** Với `EDULEAD`: p (pooled) = .019 nhưng p (Welch) = .068. Kết luận nào đáng tin hơn khi nhóm tư chỉ có 34 trường? Vì sao?

#### 📥 Đầu vào

so sánh trường CÔNG (`PRIVATESCH=1`, n=161) vs TƯ (`PRIVATESCH=2`, n=34) trên 4 chỉ số — LƯU Ý cỡ mẫu 2 nhóm CHÊNH LỆCH RẤT LỚN (161 vs 34, tỉ lệ gần 5:1), khác hẳn ví dụ cân bằng ở Module 05 của site.

#### 📤 Đầu ra thật (EDULEAD, dòng đáng chú ý nhất)

Levene_p=**0,148** (≥0,05 → về mặt kỹ thuật nên đọc dòng "pooled"/"equal variances assumed") nhưng `t_pooled=−2,356, p_pooled=0,019` (**có ý nghĩa**) trong khi `t_Welch=−1,874, p_Welch=0,068` (**KHÔNG có ý nghĩa**, dù rất sát ngưỡng 0,05) — 2 kết luận TRÁI NGƯỢC NHAU tuỳ chọn dòng nào!

#### 🎯 Trả lời câu hỏi ❓ bên dưới — kết luận nào đáng tin hơn

khi cỡ mẫu 2 nhóm CHÊNH LỆCH LỚN (161 vs 34) như ở đây, hầu hết nhà thống kê hiện đại khuyên **LUÔN dùng Welch's t-test làm mặc định an toàn** (không phụ thuộc kết quả Levene) — vì kiểm định pooled-variance vốn nhạy cảm với chênh lệch cỡ mẫu ngay cả khi Levene không có ý nghĩa (Levene bản thân cũng là 1 kiểm định có sai số, đặc biệt kém tin cậy với nhóm nhỏ n=34). Kết luận đúng nên là: **KHÔNG đủ bằng chứng khác biệt EDULEAD giữa trường công/tư (theo Welch, p=0,068)** — một minh chứng thực tế quan trọng rằng quy tắc "chỉ nhìn Levene rồi chọn dòng" (dạy ở Module 05) có giới hạn khi cỡ mẫu quá lệch, và cách an toàn hơn là ưu tiên Welch.

## 3. t bắt cặp & một mẫu

In [5]:
print("bắt cặp STAFFSHORT vs EDUSHORT:",stats.ttest_rel(truong.STAFFSHORT,truong.EDUSHORT))
for v in ["EDULEAD","ENCOURPG"]: print(v,"khác 0?",stats.ttest_1samp(truong[v],0))

bắt cặp STAFFSHORT vs EDUSHORT: TtestResult(statistic=np.float64(0.20183141415991795), pvalue=np.float64(0.8402600032491355), df=np.int64(194))
EDULEAD khác 0? TtestResult(statistic=np.float64(11.893894477001027), pvalue=np.float64(7.430419906525151e-25), df=np.int64(194))
ENCOURPG khác 0? TtestResult(statistic=np.float64(13.393334304345585), pvalue=np.float64(2.1491170439348348e-29), df=np.int64(194))


#### 📥 Đầu vào

t-test BẮT CẶP so `STAFFSHORT` (thiếu nhân sự) với `EDUSHORT` (thiếu trang thiết bị) trên CÙNG 195 trường; và 2 t-test MỘT MẪU kiểm tra xem `EDULEAD`/`ENCOURPG` có khác 0 hay không (nhớ lại: các biến WLE có trung bình QUỐC TẾ=0, nên "khác 0" nghĩa là "khác trung bình thế giới").

#### 📤 Đầu ra thật

bắt cặp STAFFSHORT vs EDUSHORT: t=0,202, p=**0,840** (không có ý nghĩa — 2 loại thiếu thốn này ở mức tương đương nhau, dù trước đó đã thấy chúng tương quan mạnh r=0,61 ở notebook buổi 1 — tương quan cao KHÔNG có nghĩa là 2 biến có cùng GIÁ TRỊ TRUNG BÌNH, chỉ là chúng biến thiên CÙNG HƯỚNG). EDULEAD khác 0: t=11,89, p=7,4e-25 (cực kỳ có ý nghĩa); ENCOURPG khác 0: t=13,39, p=2,1e-29 (cực kỳ có ý nghĩa).

#### 🎯 Diễn giải EDULEAD/ENCOURPG khác 0 có ý nghĩa

trường học Việt Nam trong mẫu PISA 2025 có mức lãnh đạo giáo dục (EDULEAD) VÀ khuyến khích phát triển chuyên môn (ENCOURPG) **CAO HƠN đáng kể so với trung bình quốc tế** (cả 2 đều mean dương lớn: 0,765 và giá trị tương ứng) — một phát hiện thực tế thú vị về đặc điểm giáo dục VN so với thế giới.

## 4. Trọng số trường — trung bình có trọng số khác trung bình thường

In [6]:
w=truong.W_NRASCHBWT; print("EDULEAD: không TS =",round(truong.EDULEAD.mean(),3),"| có TS =",round(np.average(truong.EDULEAD,weights=w),3))
print("Trọng số: min",round(w.min(),1),"max",round(w.max(),1),"→ một trường có thể 'đại diện' cho >600 trường")

EDULEAD: không TS = 0.765 | có TS = 0.787
Trọng số: min 1.0 max 617.2 → một trường có thể 'đại diện' cho >600 trường


#### 📥 Đầu vào

trọng số khảo sát `W_NRASCHBWT` (trọng số hiệu chỉnh non-response, một phần của thiết kế lấy mẫu phức hợp PISA) — so sánh trung bình THƯỜNG (mỗi trường tính như nhau) với trung bình CÓ TRỌNG SỐ (mỗi trường đóng góp theo đúng tỉ lệ nó đại diện trong tổng thể).

#### 📤 Đầu ra thật

EDULEAD không trọng số=**0,765**, có trọng số=**0,787** — chênh lệch nhỏ (~0,02) nhưng CÓ THẬT. Trọng số dao động từ **1,0 đến 617,2** — nghĩa là 1 trường trong mẫu 195 trường có thể đại diện cho tới hơn 600 trường thật ngoài tổng thể!

#### ⚠️ Bài học quan trọng về dữ liệu khảo sát quy mô lớn (large-scale assessment) như PISA

không giống dữ liệu lớp học 240 học sinh (lấy mẫu đơn giản), PISA dùng thiết kế lấy mẫu PHÂN TẦNG PHỨC HỢP — nếu tính trung bình "ngây thơ" mà bỏ qua trọng số, kết quả sẽ bị lệch (bias) theo hướng đại diện QUÁ MỨC cho các trường "dễ lấy mẫu" và ĐẠI DIỆN THIẾU cho nhóm trường có trọng số cao. Đây là lý do các báo cáo PISA chính thức LUÔN dùng trọng số khi công bố số liệu quốc gia — notebook này minh hoạ tại sao điều đó quan trọng bằng số liệu thật, không chỉ lý thuyết.

## 5. Power khi hai nhóm lệch cỡ mẫu (161 vs 34)

In [7]:
from statsmodels.stats.power import TTestIndPower
p=TTestIndPower(); print("d=0.5, 161 vs 34:",round(p.power(.5,161,.05,ratio=34/161),2),"| d=0.5, 98 vs 98:",round(p.power(.5,98,.05,ratio=1),2))

d=0.5, 161 vs 34: 0.75 | d=0.5, 98 vs 98: 0.94


#### 📥 Đầu vào

tính sẵn (power analysis) — với cỡ hiệu ứng d=0,5 (mức vừa theo Cohen), so sánh power giữa 2 kịch bản: (a) cỡ mẫu THẬT của nghiên cứu này (161 vs 34, giống ô so sánh công/tư ở trên) và (b) cỡ mẫu GIẢ ĐỊNH nếu cân bằng (98 vs 98, cùng tổng 195 trường nhưng chia đều).

#### 📤 Đầu ra thật

`d=0,5, 161 vs 34: power=0,75` nhưng `d=0,5, 98 vs 98: power=0,94` — CÙNG tổng cỡ mẫu 195 trường, nhưng cách CHIA 2 nhóm khác nhau làm power chênh nhau tới **19 điểm phần trăm** (0,75 → 0,94)!

#### 🎯 Bài học nối trực tiếp với ô [6] ở trên

đây chính xác là lý do vì sao so sánh EDULEAD công/tư (161 vs 34) cho kết quả "nhập nhằng" (p=0,019 pooled vs p=0,068 Welch) — với thiết kế lệch cỡ mẫu 161:34 như hiện tại, nghiên cứu đã "lãng phí" một phần sức mạnh thống kê so với việc thiết kế lấy mẫu cân bằng hơn ngay từ đầu (nếu có thể). Đây là bài học thiết kế nghiên cứu thực tế: khi biết trước sẽ so sánh 2 nhóm không cân bằng tự nhiên (như công/tư), nên cân nhắc lấy mẫu tăng cường (oversampling) nhóm thiểu số để đạt power tốt hơn.

## Bài tập
`bai_tap/buoi2_de.md`